# Étape 7 — Variantes du DE sur GPU (issue #17, partie GPU)

**Prêt à exécuter : Exécution → Tout exécuter.** Runtime : **GPU T4**.

Le code est dans `src/cuda/de_cuda_variantes.cu`. Il reprend exactement l'organisation de la v2 (kernel P, kernel fusionné MCER, argmin sur GPU, taille de bloc automatique) et ajoute deux stratégies et le choix de CR. La v2 elle-même n'est pas modifiée.

## Les variantes testées

| Variante | Mutation | F | CR |
|---|---|---|---|
| **rand1 CR=0,3** (référence = v2) | v = x_r1 + F (x_r2 − x_r3) | 0,5 | 0,3 |
| rand1 CR=0,1 / 0,5 / 0,9 | idem | 0,5 | 0,1 / 0,5 / 0,9 |
| **ctb1** (current-to-best/1) | v = x_i + F (x_best − x_i) + F (x_r1 − x_r2) | 0,5 | 0,3 |
| **jDE** (Brest et al., 2006) | v = x_r1 + F_i (x_r2 − x_r3) | F_i auto-adapté | CR_i auto-adapté |

- **ctb1** : x_best est le meilleur individu de la génération courante, trouvé sur GPU par `k_argmin` **avant** le kernel MCER (un lancement de plus par génération). La recherche est plus « gloutonne » : elle converge plus vite mais peut se bloquer dans un minimum local.
- **jDE** : chaque individu porte ses propres F_i et CR_i (départ : 0,5 et 0,9). À chaque génération, avec une probabilité 0,1, F' est retiré dans [0,1 ; 1] et, avec une probabilité 0,1, CR' est retiré dans [0 ; 1]. L'essai est construit avec (F', CR') ; s'il remplace x_i, les nouveaux paramètres sont gardés. F et CR sont donc eux aussi en double tampon sur le GPU.

## Protocole (grille réduite de l'issue)
D = 50, N = 100, budget 10⁴·D = 500 000 évaluations, 10 runs (graines 1 à 10), 4 fonctions, 6 variantes : 240 runs.

## Critères de validation (#17)
- **Kruskal-Wallis** entre les 6 variantes, pour chaque fonction ;
- **classement** des variantes par fonction (rang moyen).

En plus : contrôle que `rand1 CR=0,3` redonne **exactement** les résultats de la v2 (même graine → même erreur, au bit près), tests de Mann-Whitney de chaque variante contre la référence (correction de Holm), courbes de convergence et temps.

In [ ]:
from google.colab import userdata
GH_USER = "amenacer"
REPO    = "DE_CUDA"
BRANCHE = "amina/7-tester-variantes-de-gpu"
token   = userdata.get("GITHUB_TOKEN")
!rm -rf /content/{REPO}
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO}
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git checkout -q {BRANCHE} && git branch --show-current
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import subprocess, re, statistics as st, os, json
ROOT = "/content/DE_CUDA"
RAPPORT = []                       # tout ce qui est affiche est aussi garde pour le rapport
def note(s=""):
    print(s); RAPPORT.append(str(s))

def run(args, cwd=ROOT, ok_codes=(0,)):
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if p.returncode not in ok_codes:
        print(p.stdout); print(p.stderr)
        raise RuntimeError(f"{args[0]} a renvoye {p.returncode}")
    return p

def compile_cuda(src, out):
    p = run(["nvcc", "-O3", "-arch=sm_75", "-Xcompiler", "-Wall", "-Xcompiler", "-Wextra",
             "-o", out, src], cwd=f"{ROOT}/src/cuda")
    warn = (p.stdout + p.stderr).strip()
    note(f"{src} : " + ("compilation SANS aucun warning" if not warn else "WARNINGS :\n" + warn))

KV = re.compile(r"(\w+)=(\S+)")
def lance(prog, f, D, N, seed, csv=None, extra=()):
    """Lance un run et renvoie le dictionnaire de sa derniere ligne (best_error, time_s, ...)."""
    args = [prog, f, str(D), str(N), str(seed)] + ([csv] if csv else []) + list(extra)
    out = run(args).stdout.strip().splitlines()[-1]
    d = dict(KV.findall(out))
    for k in ("best_error", "time_s"):
        d[k] = float(d[k])
    return d

def mann_whitney(a, b):
    """p-valeur du test de Mann-Whitney bilateral ; 1.0 si les deux series sont identiques et constantes."""
    from scipy.stats import mannwhitneyu
    if len(set(a) | set(b)) == 1:
        return 1.0
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue)

def tableau_md(lignes, entetes):
    s = "| " + " | ".join(entetes) + " |\n|" + "---|" * len(entetes) + "\n"
    for l in lignes:
        s += "| " + " | ".join(str(x) for x in l) + " |\n"
    return s
print("outils prets")

### 1 — Compiler la v2 et les variantes

In [ ]:
compile_cuda("de_cuda_v2.cu", "de_cuda_v2")
compile_cuda("de_cuda_variantes.cu", "de_cuda_variantes")
V2  = f"{ROOT}/src/cuda/de_cuda_v2"
VAR = f"{ROOT}/src/cuda/de_cuda_variantes"
VARIANTES = [   # (nom affiche, options de la ligne de commande)
    ("rand1 CR=0,1",  ["--strategie", "rand1", "--CR", "0.1"]),
    ("rand1 CR=0,3",  ["--strategie", "rand1", "--CR", "0.3"]),   # = v2, reference
    ("rand1 CR=0,5",  ["--strategie", "rand1", "--CR", "0.5"]),
    ("rand1 CR=0,9",  ["--strategie", "rand1", "--CR", "0.9"]),
    ("ctb1 CR=0,3",   ["--strategie", "ctb1",  "--CR", "0.3"]),
    ("jDE",           ["--strategie", "jde"]),
]
REF = "rand1 CR=0,3"
FONCTIONS = ["sphere", "rosenbrock", "griewank", "rastrigin"]
D, N, SEEDS = 50, 100, range(1, 11)
print(run([VAR, "sphere", "10", "50", "1", "--strategie", "jde"]).stdout)

### 2 — Contrôle : `rand1 CR=0,3` = v2 au bit près

Les deux programmes partagent le même code de tirage aléatoire ; avec la même graine ils doivent donner exactement la même erreur finale. C'est la preuve que le nouveau code ne change rien à la v2 quand on choisit ses paramètres.

In [ ]:
lignes, tout_ok = [], True
for f in FONCTIONS:
    for seed in (1, 2, 3):
        a = lance(V2, f, D, N, seed)["best_error"]
        b = lance(VAR, f, D, N, seed, extra=["--strategie", "rand1", "--CR", "0.3"])["best_error"]
        tout_ok &= (a == b)
        lignes.append([f, seed, f"{a:.9e}", f"{b:.9e}", "identique" if a == b else "DIFFERENT"])
note(f"Controle v2 contre variantes (rand1, F = 0,5, CR = 0,3), D = {D}, N = {N}")
note(tableau_md(lignes, ["fonction", "graine", "v2", "variantes rand1", "verdict"]))
note("=> " + ("OK : resultats identiques au bit pres" if tout_ok else "ECHEC : les resultats different"))
assert tout_ok

### 3 — Campagne : 4 fonctions × 6 variantes × 10 runs

In [ ]:
os.makedirs(f"{ROOT}/results", exist_ok=True)
CSV_VAR = f"{ROOT}/results/variantes_gpu_D50_N100.csv"
if os.path.exists(CSV_VAR): os.remove(CSV_VAR)
ERR, TPS, JDE_PARAM = {}, {}, {}
for f in FONCTIONS:
    for nom, opts in VARIANTES:
        ERR[(f, nom)], TPS[(f, nom)] = [], []
        for seed in SEEDS:
            d = lance(VAR, f, D, N, seed, CSV_VAR, extra=opts)
            assert d["sante"] == "OK", d
            ERR[(f, nom)].append(d["best_error"]); TPS[(f, nom)].append(d["time_s"])
            if nom == "jDE":
                JDE_PARAM.setdefault(f, []).append((float(d["F_moyen"]), float(d["CR_moyen"])))
    print(f, "fait")
print("lignes CSV :", sum(1 for _ in open(CSV_VAR)) - 1)

### 4 — Qualité, Kruskal-Wallis et classement

- **Kruskal-Wallis** : les 6 variantes donnent-elles la même distribution d'erreurs ? (p < 0,05 → au moins une variante diffère.)
- **Rang moyen** : on classe ensemble les 60 erreurs d'une fonction (rang 1 = la plus petite) et on fait la moyenne des rangs de chaque variante. Rang moyen faible = meilleure variante. Les ex æquo reçoivent le rang moyen.
- **Mann-Whitney contre la référence**, p-valeurs corrigées par **Holm** (5 comparaisons par fonction).

In [ ]:
from scipy.stats import kruskal, rankdata

def holm(ps):
    """p-valeurs ajustees de Holm (dans l'ordre d'entree)."""
    m = len(ps); ordre = sorted(range(m), key=lambda k: ps[k])
    adj, courant = [0.0] * m, 0.0
    for rang, k in enumerate(ordre):
        courant = max(courant, min(1.0, (m - rang) * ps[k]))
        adj[k] = courant
    return adj

noms = [n for n, _ in VARIANTES]
RANG = {}
for f in FONCTIONS:
    series = [ERR[(f, n)] for n in noms]
    valeurs = [v for s in series for v in s]
    if len(set(valeurs)) == 1:
        p_kw, h = 1.0, 0.0
    else:
        h, p_kw = kruskal(*series)
    rangs = rankdata(valeurs)
    for k, n in enumerate(noms):
        RANG[(f, n)] = float(st.mean(rangs[k * len(SEEDS):(k + 1) * len(SEEDS)]))
    autres = [n for n in noms if n != REF]
    p_mw = holm([mann_whitney(ERR[(f, REF)], ERR[(f, n)]) for n in autres])
    p_mw = dict(zip(autres, p_mw))
    classement = sorted(noms, key=lambda n: RANG[(f, n)])
    lignes = []
    for n in classement:
        e = ERR[(f, n)]
        lignes.append([classement.index(n) + 1, n, f"{st.mean(e):.3e} ± {st.pstdev(e):.2e}", f"{st.median(e):.3e}",
                       f"{RANG[(f, n)]:.1f}",
                       "reference" if n == REF else f"{p_mw[n]:.3g}" + (" *" if p_mw[n] < 0.05 else "")])
    note(f"{f} (D = {D}, N = {N}, 10 runs) -- Kruskal-Wallis : H = {h:.2f}, p = {p_kw:.3g}"
         + (" => difference significative entre variantes" if p_kw < 0.05 else " => pas de difference detectee"))
    note(tableau_md(lignes, ["rang", "variante", "moyenne ± ET", "mediane", "rang moyen", "p Holm vs reference"]))
lignes = [[n] + [f"{RANG[(f, n)]:.1f}" for f in FONCTIONS] + [f"{st.mean(RANG[(f, n)] for f in FONCTIONS):.2f}"]
          for n in sorted(noms, key=lambda n: st.mean(RANG[(f, n)] for f in FONCTIONS))]
note("Classement global (rang moyen par fonction, puis moyenne des 4 fonctions ; plus petit = meilleur)")
note(tableau_md(lignes, ["variante"] + FONCTIONS + ["moyenne"]))
note("* : p Holm < 0,05. Les ecarts-types sont des ecarts-types de population (statistics.pstdev).")

### 5 — Ce que jDE a appris

Moyenne, sur la population finale, des F_i et CR_i (puis moyenne sur les 10 runs). Départ : F = 0,5, CR = 0,9.

In [ ]:
lignes = [[f, f"{st.mean(p[0] for p in JDE_PARAM[f]):.3f}", f"{st.mean(p[1] for p in JDE_PARAM[f]):.3f}"] for f in FONCTIONS]
note("jDE : parametres moyens en fin de run (10 runs)")
note(tableau_md(lignes, ["fonction", "F moyen", "CR moyen"]))

### 6 — Temps par variante

ctb1 lance un kernel de plus par génération (argmin) ; jDE lit et écrit F et CR en plus.

In [ ]:
lignes = [[n] + [f"{st.mean(TPS[(f, n)]):.4f}" for f in FONCTIONS] for n in noms]
note(f"Temps moyen (s) par run, D = {D}, N = {N}, 10 runs")
note(tableau_md(lignes, ["variante"] + FONCTIONS))

### 7 — Courbes de convergence (graine 1)

`--trace` affiche la meilleure erreur à chaque génération. Le mode trace ajoute un argmin et une copie par génération : ces runs ne servent qu'aux courbes, pas aux temps.

In [ ]:
import matplotlib.pyplot as plt
os.makedirs(f"{ROOT}/figures", exist_ok=True)
TR = re.compile(r"TRACE gen=\d+ FE=(\d+) best_error=(\S+)")
fig, axs = plt.subplots(2, 2, figsize=(10, 7))
for ax, f in zip(axs.flat, FONCTIONS):
    for nom, opts in VARIANTES:
        out = run([VAR, f, str(D), str(N), "1", "--trace"] + opts).stdout
        pts = [(int(a), float(b)) for a, b in TR.findall(out)]
        ax.plot([p[0] for p in pts], [max(p[1], 1e-30) for p in pts], label=nom, lw=1.2)
    ax.set_yscale("log"); ax.set_title(f"{f} (D = {D}, N = {N})")
    ax.set_xlabel("evaluations"); ax.set_ylabel("meilleure erreur")
axs[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Variantes du DE sur GPU T4, graine 1 (erreur 0 tracee a 1e-30)"); fig.tight_layout()
fig.savefig(f"{ROOT}/figures/convergence_variantes_D50.png", dpi=200); plt.show()

### 8 — Sauvegarder et pousser

In [ ]:
os.makedirs(f"{ROOT}/results", exist_ok=True)
with open(f"{ROOT}/results/etape7_variantes_gpu.md", "w") as fp:
    fp.write("# Étape 7 — Variantes du DE sur GPU (issue #17, partie GPU)\n\n")
    fp.write("GPU : " + run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).stdout.strip() + "\n\n")
    fp.write("```\n" + "\n".join(RAPPORT) + "\n```\n")
%cd /content/DE_CUDA
!git add src/cuda/de_cuda_variantes.cu results/variantes_gpu_D50_N100.csv figures/convergence_variantes_D50.png results/etape7_variantes_gpu.md
!git status --short
!git commit -q -m "7 : variantes du DE sur GPU (rand1 CR, current-to-best, jDE), Kruskal-Wallis et classement (refs #17)" && git push -q origin {BRANCHE} && echo "=> pousse sur {BRANCHE}"